# Repeating ProD with k=(Total Number of Classes)=6

In [1]:
import os, sys
import numpy as np
import pandas as pd
from pathlib import Path
from time import process_time

from prod_fs import ProD

# According to Canedo (2012), 3 % of 561 features = 17 features
nRetainedFeatures = 17

Xtrain = pd.read_csv(
    Path("data") / "train" / "X_train.txt", sep=r'\s+', header=None
).values
ytrain = pd.read_csv(
    Path("data") / "train" / "y_train.txt", sep=r'\s+',
    header=None, index_col=None
)
ytrain = np.reshape(ytrain, -1)

Xtest = pd.read_csv(
    Path("data") / "test" / "X_test.txt", sep=r'\s+', header=None
).values
ytest = pd.read_csv(
    Path("data") / "test" / "y_test.txt", sep=r'\s+',
    header=None, index_col=None
)
ytest = np.reshape(ytest, -1)

fsorder = [
    ("ProD-Sco", "customScott", "mean"), 
    ("ProD-Slv", "customSilverman", "mean"), 
    ("ProD-ScoWeighted", "customScott", "weighted"), 
    ("ProD-SlvWeighted", "customSilverman", "weighted")
]

results = pd.DataFrame(
    data=np.zeros((4, 5)), index=[fs[0] for fs in fsorder],
    columns=["kNN", "SVM", "Gaussian-NB", "LDA", "DT"]
)

In [2]:
rankers = []
time = []
for fs in fsorder:
    tProD_start = process_time()
    # Carry out FS
    prodRanker = ProD(
        integration_method="trapz", delta=500, bw_method=fs[1],
        k=6, n_jobs=-1, mode="release", lower_end=-1.5, upper_end=2.5,
        averaging_method=fs[2]
    )
    prodRanker.fit(Xtrain, ytrain)
    tProD_stop = process_time()
    tProD = tProD_stop - tProD_start
    
    rankers.append(prodRanker)
    time.append(tProD)

Constructing and evaluating the KDEs per class for every feature ... 
 - Kernels constructed!
Computing intersection areas ...
 - averaging_method: 'mean'
Constructing and evaluating the KDEs per class for every feature ... 
 - Kernels constructed!
Computing intersection areas ...
 - averaging_method: 'mean'
Constructing and evaluating the KDEs per class for every feature ... 
 - Kernels constructed!
Computing intersection areas ...
 - averaging_method: 'weighted'
Constructing and evaluating the KDEs per class for every feature ... 
 - Kernels constructed!
Computing intersection areas ...
 - averaging_method: 'weighted'


In [4]:
for ranker in rankers:
    print(ranker.get_topnFeatures(nRetainedFeatures))

[56, 52, 40, 558, 271, 3, 268, 6, 265, 9, 226, 231, 95, 359, 234, 86, 233]
[56, 52, 40, 9, 271, 558, 3, 265, 6, 268, 95, 226, 231, 366, 359, 99, 86]
[56, 52, 40, 558, 271, 3, 268, 6, 265, 9, 226, 231, 95, 359, 234, 86, 233]
[56, 52, 40, 9, 271, 558, 3, 265, 6, 268, 95, 226, 231, 366, 359, 99, 86]


In [6]:
# Train-Test
results = pd.DataFrame(
    data=np.zeros((len(fsorder), 5)), index=[fs[0] for fs in fsorder],
    columns=["kNN", "SVM", "Gaussian-NB", "LDA", "DT"]
)
results

,kNN,SVM,Gaussian-NB,LDA,DT
ProD-Sco,0.0,0.0,0.0,0.0,0.0
ProD-Slv,0.0,0.0,0.0,0.0,0.0
ProD-ScoWeighted,0.0,0.0,0.0,0.0,0.0
ProD-SlvWeighted,0.0,0.0,0.0,0.0,0.0


In [9]:
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV

from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from sklearn.metrics import balanced_accuracy_score

for i, fs in enumerate(fsorder):
    print(f" - Feature selection scheme: {fs[0]}")
    top_features = rankers[i].get_topnFeatures(nRetainedFeatures)

    Xtrain_reduced = Xtrain[:, top_features]
    Xtest_reduced  = Xtest[:, top_features]

    # 3-fold grid search cross-validation
    CV_3fold = KFold(n_splits=3, shuffle=True, random_state=0)

    # kNN
    kNN = KNeighborsClassifier(weights="uniform")
    kNN_params = {'n_neighbors': [5,7,9]}
    clfkNN_GS = GridSearchCV(
        kNN, kNN_params, cv=CV_3fold, n_jobs=-1, scoring="balanced_accuracy"
    )
    clfkNN_GS.fit(Xtrain_reduced, ytrain)
    results.at[fs[0], "kNN"] = balanced_accuracy_score(
        ytest, clfkNN_GS.predict(Xtest_reduced)
    )
    
    # SVM
    svm_clf = SVC()
    svm_params = {'C': [1,10,100,1000], 'gamma': [0.001,0.0001], 'kernel': ['rbf']}
    clfsvm_GS = GridSearchCV(
        svm_clf, svm_params, cv=CV_3fold, n_jobs=-1, scoring="balanced_accuracy"
    )
    clfsvm_GS.fit(Xtrain_reduced, ytrain)
    results.at[fs[0], "SVM"] = balanced_accuracy_score(
        ytest, clfsvm_GS.predict(Xtest_reduced)
    )
    
    # Gaussian Naive-Bayes
    naiveBayesClf = GaussianNB()
    naiveBayesClf.fit(Xtrain_reduced, ytrain)
    results.at[fs[0], "Gaussian-NB"] = balanced_accuracy_score(
        ytest, naiveBayesClf.predict(Xtest_reduced)
    )
    
    # LDA
    ldaClf = LinearDiscriminantAnalysis()
    ldaClf.fit(Xtrain_reduced, ytrain)
    results.at[fs[0], "LDA"] = balanced_accuracy_score(
        ytest, ldaClf.predict(Xtest_reduced)
    )

    # DT
    dt_clf = DecisionTreeClassifier(random_state=0)
    dt_params = {'splitter': ["best", "random"], 'max_depth': [3,5,7,9]}
    clfdt_GS = GridSearchCV(
        dt_clf, dt_params, cv=CV_3fold, scoring="balanced_accuracy"
    )
    clfdt_GS.fit(Xtrain_reduced, ytrain)
    results.at[fs[0], "DT"] = balanced_accuracy_score(
        ytest, clfdt_GS.predict(Xtest_reduced)
    )

 - Feature selection scheme: ProD-Sco
 - Feature selection scheme: ProD-Slv
 - Feature selection scheme: ProD-ScoWeighted
 - Feature selection scheme: ProD-SlvWeighted


In [10]:
results

,kNN,SVM,Gaussian-NB,LDA,DT
ProD-Sco,0.827823,0.863395,0.740073,0.838090,0.799494
ProD-Slv,0.820469,0.862454,0.718327,0.827881,0.801187
ProD-ScoWeighted,0.827823,0.863395,0.740073,0.838090,0.799494
ProD-SlvWeighted,0.820469,0.862454,0.718327,0.827881,0.801187


In [11]:
results.to_csv("ProD_kAll.csv")